# Basin Extraction — Spatial Mapping of Significant Terrain Depressions

## Objective

The previous notebooks established *that* topologically significant basins exist and *how many* there are. This notebook answers **where** they are: each high-persistence H₀ pair is mapped back to its geographic location and spatial extent.

We work exclusively on **imputed data** — the complete terrain reconstructed in `TDA_Data_Imputation.ipynb`. Using raw data would leave 63% of the Amsterdam AOI and 27% of the Eindhoven AOI undefined, making spatial basin delineation unreliable.

---

## From persistence pairs to spatial basins

Each finite H₀ pair $(b, d)$ produced by `gudhi.CubicalComplex` corresponds to one basin:

- **Birth cell** — the pixel at elevation $b$, i.e. the local minimum where the basin first appears in the filtration. This is the deepest point of the depression.
- **Death cell** — the pixel (saddle) at elevation $d$ where the basin overflows into a lower neighbour and the connected component merges. This is the spillover point.
- **Basin extent** — the connected component of $\{f \le d\}$ that contains the birth cell. Everything that would flood before the basin overflows.
- **Persistence** $p = d - b$ — the spillover depth in metres. Only pairs with $p > \tau$ (our threshold) are considered meaningful.

To retrieve the birth and death cell coordinates we call `gudhi`'s `cofaces_of_persistence_pairs()` after `compute_persistence()`. This returns flat cell indices which we convert back to (row, col) pixel coordinates.

---

## Notebook structure

| Section | Content |
|---|---|
| **1 — Imports + data loading** | Load imputed elevation crops and pre-computed TDA results from `02_comparison.ipynb` |
| **2 — Threshold selection** | Justify the persistence threshold $\tau$ from the sorted-persistence elbow |
| **3 — Basin extraction** | Retrieve birth/death cell coordinates, flood-fill basin extents, visualise spatially |
| **4 — Statistics + comparison** | Basin area, volume, depth per city — side-by-side summary |

In [ ]:
# Cell 1 — imports
from pathlib import Path
import warnings

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch

import rasterio
from rasterio.transform import Affine
from rasterio.windows import from_bounds as window_from_bounds

import gudhi as gd
from scipy import ndimage

print(f"numpy    {np.__version__}")
print(f"rasterio {rasterio.__version__}")
print(f"gudhi    {gd.__version__}")

In [ ]:
# Cell 2 — paths and constants
#
# All data paths follow the repo-relative convention introduced in
# TDA_Data_Imputation.ipynb and used throughout 01 and 02.

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "requirements.txt").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "requirements.txt").exists():
    raise FileNotFoundError(
        "requirements.txt not found. Make sure the kernel is running inside the repo."
    )

DATA_DIR  = REPO_ROOT / "data"
OUT_DIR   = REPO_ROOT / "outputs"

# Raw .tif files — used only to read the affine transform / CRS metadata.
# No pixel data is read from them in this notebook.
RAW_TIF = {
    "amsterdam": DATA_DIR / "amsterdam_centre_strip.tif",
    "eindhoven": DATA_DIR / "eindhoven_north_strip.tif",
}

# Fully imputed rasters saved by TDA_Data_Imputation.ipynb
IMPUTED_NPY = {
    "amsterdam": REPO_ROOT / "output"           / "ams_final_imputed.npy",
    "eindhoven": REPO_ROOT / "imputed_datasets" / "ehv_final_imputed.npy",
}

# Pre-computed TDA results on imputed data saved by 02_comparison.ipynb
TDA_NPZ = {
    "amsterdam": OUT_DIR / "persistence_amsterdam_imputed_comparison.npz",
    "eindhoven": OUT_DIR / "persistence_eindhoven_imputed_comparison.npz",
}

# Zoom bounding boxes — identical to 01 and 02 (RD New, EPSG:28992)
ZOOM_BBOX = {
    "amsterdam": (121_500, 486_500, 122_500, 487_500),
    "eindhoven": (161_500, 383_500, 162_500, 384_500),
}

# Native resolution of AHN DTM
RES = 0.5   # metres per pixel

# Verify all required files exist
print("File check:")
for label, path in {**TDA_NPZ, **IMPUTED_NPY}.items():
    status = "OK" if path.exists() else "MISSING"
    print(f"  [{status}]  {path.name}")

print(f"\nREPO_ROOT = {REPO_ROOT}")

In [ ]:
# Cell 3 — loader helpers
#
# load_npy_window : memory-map slice from a full-raster .npy.
#                  Uses the matching .tif only for transform/CRS — no pixel I/O.
# reconstruct_transform : rebuild a rasterio Affine from the 6-element vector
#                         stored in the .npz files.

NODATA_MAGNITUDE = 1e30

def load_npy_window(npy_path, tif_path, bbox):
    """Slice a bounding-box crop from a full-raster .npy via memory map."""
    xmin, ymin, xmax, ymax = bbox
    with rasterio.open(tif_path) as src:
        win    = window_from_bounds(xmin, ymin, xmax, ymax, src.transform)
        r_off  = int(win.row_off)
        c_off  = int(win.col_off)
        height = int(win.height)
        width  = int(win.width)
        transform = src.window_transform(win)
        crs    = src.crs
        nodata = src.nodata

    arr_mmap = np.load(npy_path, mmap_mode="r")
    crop = arr_mmap[r_off:r_off + height, c_off:c_off + width].copy()
    del arr_mmap

    crop = crop.astype(np.float32, copy=False)
    if nodata is not None and np.isfinite(nodata):
        crop[crop == np.float32(nodata)] = np.nan
    crop[np.abs(crop) > NODATA_MAGNITUDE] = np.nan
    return crop, transform, crs


def reconstruct_transform(tr_vec):
    """Rebuild a rasterio Affine from the 6-element vector saved in .npz files."""
    return Affine(tr_vec[0], tr_vec[1], tr_vec[2],
                  tr_vec[3], tr_vec[4], tr_vec[5])

In [ ]:
# Cell 4 — load imputed crops and TDA results
#
# Elevation arrays: loaded fresh from the .npy files via mmap (~32 MB each).
# TDA results: loaded from the .npz files saved by 02_comparison.ipynb —
#              no need to re-run gudhi.
#
# Everything is stored in `data[city]` for clean downstream access.

data = {}

for city in ("amsterdam", "eindhoven"):
    elev, transform, crs = load_npy_window(
        IMPUTED_NPY[city], RAW_TIF[city], ZOOM_BBOX[city]
    )

    npz = np.load(TDA_NPZ[city])
    h0  = npz["h0"]   # shape (N, 2) — columns: [birth, death]

    fin  = np.isfinite(h0[:, 1])
    pers = h0[fin, 1] - h0[fin, 0]

    data[city] = {
        "elev":      elev,
        "transform": transform,
        "crs":       crs,
        "h0":        h0,
    }

    print(f"{city.capitalize()}:")
    print(f"  elev shape     {elev.shape}   NaN: {np.isnan(elev).mean()*100:.2f}%")
    print(f"  elev range     [{np.nanmin(elev):.2f}, {np.nanmax(elev):.2f}] m NAP")
    print(f"  H0 pairs       {len(h0):,}   (finite: {fin.sum():,})")
    print(f"  max persistence {pers.max():.3f} m")
    print()

## Threshold Selection

Running sublevel-set persistence on a 2000 × 2000 grid produces tens of thousands of H₀ pairs — most of them correspond to micro-scale roughness (sensor noise, pavement texture, IDW boundary artefacts) rather than real terrain depressions. We need a persistence threshold $\tau$ to separate **signal** from **noise**.

### The elbow method on sorted-persistence curves

Plotting pairs by decreasing persistence on a log-log scale reveals a characteristic shape:

- A **flat plateau** at high persistence (rank 1–~100): a small number of deep, long-lived basins. These are structurally meaningful depressions — canal sections, underpasses, polder corners, river channels.
- A **steep drop** at intermediate persistence: features that exist but are shallow.
- A **noise floor** near the bottom (persistence < 0.01 m): floating-point rounding artefacts and sub-pixel roughness.

The **elbow** — where the curve transitions from plateau to steep drop — is the natural threshold candidate. From `02_comparison.ipynb` we already observed this transition near **0.1 m** for both cities.

### Physical interpretation of $\tau = 0.1$ m

A basin survives the threshold if and only if it can hold at least 10 cm of water before spilling into a neighbour. At 0.5 m pixel resolution this filters out roughness at the sub-decimetre scale while retaining features that would matter in a real inundation scenario.

In [ ]:
# Cell 5 — sorted-persistence curves with candidate threshold lines
#
# One subplot per city. Horizontal lines mark four candidate thresholds
# so we can read off visually how many basins survive each one.

CANDIDATES = [0.05, 0.10, 0.50, 1.00]   # metres
COLORS     = ["#aaaaaa", "#e07b00", "#cc0000", "#7700cc"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor="white")
fig.suptitle("Sorted H₀ Persistence — Imputed Data\n"
             "(horizontal lines = candidate thresholds)",
             fontsize=12, fontweight="bold")

for ax, city in zip(axes, ("amsterdam", "eindhoven")):
    h0   = data[city]["h0"]
    fin  = np.isfinite(h0[:, 1])
    pers = np.sort(h0[fin, 1] - h0[fin, 0])[::-1]

    ax.plot(np.arange(1, len(pers) + 1), pers,
            lw=1.4, color="C0", label=f"imputed (n={len(pers):,})")

    for tau, col in zip(CANDIDATES, COLORS):
        n_above = int((pers >= tau).sum())
        ax.axhline(tau, color=col, lw=1.1, ls="--",
                   label=f"τ = {tau} m  →  {n_above:,} basins")

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("rank", fontsize=10)
    ax.set_ylabel("persistence (m)", fontsize=10)
    ax.set_title(city.capitalize(), fontsize=11)
    ax.legend(fontsize=8, loc="upper right")
    ax.grid(True, which="both", alpha=0.25)

plt.tight_layout()
plt.savefig(OUT_DIR / "step5_threshold_selection.png", dpi=150,
            facecolor="white", bbox_inches="tight")
print(f"Saved → {OUT_DIR / 'step5_threshold_selection.png'}")
plt.show()

In [ ]:
# Cell 6 — threshold sweep table + define THRESHOLD
#
# Print how many basins survive at each candidate threshold for both cities,
# then set THRESHOLD to the chosen value for use in all subsequent cells.

print(f"{'Threshold (m)':<16} {'Amsterdam':>12} {'Eindhoven':>12}")
print("-" * 42)

for tau in CANDIDATES:
    counts = []
    for city in ("amsterdam", "eindhoven"):
        h0   = data[city]["h0"]
        fin  = np.isfinite(h0[:, 1])
        pers = h0[fin, 1] - h0[fin, 0]
        counts.append(int((pers >= tau).sum()))
    print(f"  τ = {tau:.2f} m      {counts[0]:>10,}   {counts[1]:>10,}")

print()

# Chosen threshold — justified by the elbow in the sorted-persistence curves.
# A basin must hold at least this depth of water before spilling to be considered
# topologically significant. Change this value to explore different cutoffs.
THRESHOLD = 0.10   # metres

print(f"Selected threshold: τ = {THRESHOLD} m")
for city in ("amsterdam", "eindhoven"):
    h0   = data[city]["h0"]
    fin  = np.isfinite(h0[:, 1])
    pers = h0[fin, 1] - h0[fin, 0]
    n    = int((pers >= THRESHOLD).sum())
    print(f"  {city.capitalize()}: {n:,} significant basins out of {fin.sum():,} finite pairs")

## Basin Extraction — Spatial Mapping

### Method

For each significant H₀ pair $(b, d)$ with persistence $p = d - b \ge \tau$ we need two things:

1. **Birth location** — the pixel that is the local minimum of the basin (where the connected component was born). We find all strict 8-neighbour local minima in the elevation array and match each pair to the minimum whose elevation is closest to $b$.

2. **Basin extent** — the connected component of the sublevel set $\{f \le d\}$ that contains the birth pixel. Implemented via `scipy.ndimage.label` on the binary mask `elev <= d`, then selecting the label at the birth location.

The output is a **persistence map**: a 2D array where each pixel holds the persistence value of the deepest significant basin that covers it (zero where no significant basin reaches). This lets us visualise which zones of the terrain are topologically the most relevant for flood risk.

In [ ]:
# Cell 7 — basin extraction function

def build_basin_map(elev, h0, threshold):
    """
    Map each significant H0 pair back to its spatial location and extent.

    Parameters
    ----------
    elev      : 2D float array — imputed elevation (NaN-free)
    h0        : (N, 2) array  — H0 persistence pairs [birth, death]
    threshold : float         — minimum persistence to keep (metres)

    Returns
    -------
    pers_map  : 2D float32 array — each pixel = persistence of the deepest
                significant basin covering it (0 where no basin reaches)
    birth_pts : list of (row, col, persistence) for each mapped basin
    """
    H, W = elev.shape

    # --- Find all strict 8-neighbour local minima --------------------------
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        min_filt = ndimage.minimum_filter(
            elev, size=3, mode="constant", cval=np.inf
        )
    local_min_mask = (elev == min_filt) & np.isfinite(elev)
    min_positions  = np.argwhere(local_min_mask)        # (M, 2)
    min_elevations = elev[local_min_mask]               # (M,)

    # --- Select significant pairs -------------------------------------------
    fin      = np.isfinite(h0[:, 1])
    pers_all = h0[fin, 1] - h0[fin, 0]
    sig_h0   = h0[fin][pers_all >= threshold]
    sig_pers = pers_all[pers_all >= threshold]

    # Sort by persistence descending so high-persistence basins paint last
    # (they overwrite low-persistence ones in the map)
    order    = np.argsort(sig_pers)
    sig_h0   = sig_h0[order]
    sig_pers = sig_pers[order]

    # --- Extract basin extents ---------------------------------------------
    pers_map  = np.zeros((H, W), dtype=np.float32)
    birth_pts = []

    for i, ((b, d), p) in enumerate(zip(sig_h0, sig_pers)):
        if i % 300 == 0:
            print(f"  {i+1:>5}/{len(sig_h0)}  ...", end="\r", flush=True)

        # Match pair to closest local minimum by elevation
        best     = np.argmin(np.abs(min_elevations - b))
        br, bc   = min_positions[best]

        # Flood-fill: connected component of {f <= d} containing (br, bc)
        below    = (elev <= d) & np.isfinite(elev)
        labeled, _ = ndimage.label(below)
        lbl      = labeled[br, bc]
        if lbl == 0:
            continue

        basin_mask = labeled == lbl

        # Paint: higher persistence overwrites lower
        pers_map[basin_mask & (p > pers_map)] = p
        birth_pts.append((br, bc, float(p)))

    print(f"  Done — {len(birth_pts)} basins mapped.              ")
    return pers_map, birth_pts

In [ ]:
# Cell 8 — run extraction for both cities
# This cell may take 30–60 s per city depending on hardware.

results = {}

for city in ("amsterdam", "eindhoven"):
    print(f"{city.capitalize()} (τ = {THRESHOLD} m) ...")
    elev = data[city]["elev"].astype(np.float32)
    h0   = data[city]["h0"]

    pers_map, birth_pts = build_basin_map(elev, h0, THRESHOLD)

    results[city] = {
        "pers_map":  pers_map,
        "birth_pts": birth_pts,
    }

    covered = (pers_map > 0).sum()
    print(f"  Pixels covered by significant basins: {covered:,} "
          f"({100 * covered / elev.size:.1f}% of AOI)\n")

In [ ]:
# Cell 9 — spatial map: terrain + basin overlay + birth points
#
# Layout: 1 row × 2 columns (Amsterdam | Eindhoven).
# Background  : terrain heatmap (gist_earth, downsampled 2x for display).
# Basin layer : persistence map overlaid in semi-transparent YlOrRd —
#               yellow = low persistence, red = high persistence.
# Birth points: white dots marking each basin's deepest pixel.

fig, axes = plt.subplots(1, 2, figsize=(16, 8), facecolor="white")
fig.suptitle(f"Significant Terrain Basins (τ = {THRESHOLD} m)\n"
             "colour = persistence (spillover depth); dots = basin minima",
             fontsize=13, fontweight="bold", y=1.01)

cmap_basin = plt.get_cmap("YlOrRd")

for ax, city in zip(axes, ("amsterdam", "eindhoven")):
    elev      = data[city]["elev"]
    pers_map  = results[city]["pers_map"]
    birth_pts = results[city]["birth_pts"]

    # --- Background terrain (downsampled 2x) ---
    vis = 2
    terrain = elev[::vis, ::vis]
    ax.imshow(terrain, cmap="gist_earth",
              vmin=np.nanpercentile(elev, 1),
              vmax=np.nanpercentile(elev, 99),
              interpolation="nearest", alpha=0.7)

    # --- Basin overlay (masked where no basin) ---
    pmap_vis   = pers_map[::vis, ::vis]
    basin_rgba = cmap_basin(
        plt.Normalize(vmin=THRESHOLD, vmax=pers_map.max())(pmap_vis)
    )
    basin_rgba[..., 3] = np.where(pmap_vis > 0, 0.75, 0.0)   # alpha mask
    ax.imshow(basin_rgba, interpolation="nearest")

    # --- Birth points ---
    if birth_pts:
        rows = np.array([r for r, c, p in birth_pts]) / vis
        cols = np.array([c for r, c, p in birth_pts]) / vis
        ax.scatter(cols, rows, s=6, c="white", linewidths=0,
                   alpha=0.85, zorder=5)

    # --- Colorbar ---
    sm = plt.cm.ScalarMappable(
        cmap=cmap_basin,
        norm=plt.Normalize(vmin=THRESHOLD, vmax=pers_map.max())
    )
    sm.set_array([])
    cb = fig.colorbar(sm, ax=ax, fraction=0.046, pad=0.04, shrink=0.85)
    cb.set_label("persistence (m)", fontsize=9)

    n_basins = len(birth_pts)
    ax.set_title(f"{city.capitalize()} — {n_basins:,} basins", fontsize=11)
    ax.axis("off")

plt.tight_layout()
plt.savefig(OUT_DIR / "step6_basin_map.png", dpi=150,
            facecolor="white", bbox_inches="tight")
print(f"Saved → {OUT_DIR / 'step6_basin_map.png'}")
plt.show()

## Basin Statistics & Cross-City Comparison

With the spatial maps computed we can now quantify the topological structure of each city.

We derive three complementary views:

1. **Summary table** — basin count, AOI coverage, and persistence statistics per city.
2. **Persistence histogram** — how are the significant basins distributed across depth values? A heavy tail towards high persistence means few but very deep basins dominate; a steep drop means most basins are near the threshold.
3. **Coverage curve** — for each threshold $\tau'$, what fraction of the AOI is covered by basins with persistence $\ge \tau'$? This tells us how much terrain would be "at risk" under increasingly selective filtering.

In [ ]:
# Cell 10 — summary table
#
# All statistics are derived from already-computed data (birth_pts, pers_map)
# so this cell runs instantly — no flood-fill re-computation needed.

PIXEL_AREA = 0.5 ** 2   # m² per pixel at 0.5 m resolution

print(f"{'':18} {'Amsterdam':>14} {'Eindhoven':>14}")
print("-" * 48)

for label, key in [
    ("Significant basins",   "n_basins"),
    ("AOI coverage",         "coverage"),
    ("Max persistence (m)",  "max_pers"),
    ("P75 persistence (m)",  "p75_pers"),
    ("Median pers. (m)",     "med_pers"),
    ("Mean pers. (m)",       "mean_pers"),
    ("Covered area (m²)",    "covered_area"),
]:
    row = []
    for city in ("amsterdam", "eindhoven"):
        pts      = results[city]["birth_pts"]
        pmap     = results[city]["pers_map"]
        pers_arr = np.array([p for _, _, p in pts])
        covered  = (pmap > 0).sum()

        if key == "n_basins":
            row.append(f"{len(pts):>12,}")
        elif key == "coverage":
            row.append(f"{100 * covered / pmap.size:>11.1f}%")
        elif key == "max_pers":
            row.append(f"{pers_arr.max():>13.3f}")
        elif key == "p75_pers":
            row.append(f"{np.percentile(pers_arr, 75):>13.3f}")
        elif key == "med_pers":
            row.append(f"{np.median(pers_arr):>13.3f}")
        elif key == "mean_pers":
            row.append(f"{pers_arr.mean():>13.3f}")
        elif key == "covered_area":
            row.append(f"{covered * PIXEL_AREA:>11,.0f}")

    print(f"  {label:<18} {'  '.join(row)}")

In [ ]:
# Cell 11 — persistence histogram + coverage curve
#
# Left : histogram of persistence values for significant basins (both cities).
#        Log x-axis to see the full range from τ to max persistence.
# Right: coverage curve — fraction of AOI covered by basins with p >= tau_prime,
#        computed from the persistence map for each city.

fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor="white")
fig.suptitle("Basin Statistics — Amsterdam vs Eindhoven",
             fontsize=12, fontweight="bold")

colors = {"amsterdam": "C0", "eindhoven": "C1"}

# --- Left: persistence histogram ---
ax = axes[0]
bins = np.logspace(np.log10(THRESHOLD), np.log10(6.0), 40)

for city in ("amsterdam", "eindhoven"):
    pers_arr = np.array([p for _, _, p in results[city]["birth_pts"]])
    ax.hist(pers_arr, bins=bins, alpha=0.6,
            color=colors[city], label=city.capitalize(),
            edgecolor="white", linewidth=0.3)

ax.axvline(THRESHOLD, color="k", lw=1.2, ls="--", label=f"τ = {THRESHOLD} m")
ax.set_xscale("log")
ax.set_xlabel("persistence (m)", fontsize=10)
ax.set_ylabel("basin count", fontsize=10)
ax.set_title("Persistence distribution of significant basins", fontsize=10)
ax.legend(fontsize=9)
ax.grid(True, which="both", alpha=0.25)

# --- Right: coverage curve ---
ax = axes[1]
tau_range = np.logspace(np.log10(THRESHOLD), np.log10(5.5), 80)

for city in ("amsterdam", "eindhoven"):
    pmap   = results[city]["pers_map"]
    n_pix  = pmap.size
    coverage = [(pmap >= t).sum() / n_pix * 100 for t in tau_range]
    ax.plot(tau_range, coverage,
            color=colors[city], lw=1.8, label=city.capitalize())

ax.axvline(THRESHOLD, color="k", lw=1.2, ls="--", label=f"τ = {THRESHOLD} m")
ax.set_xscale("log")
ax.set_xlabel("threshold τ' (m)", fontsize=10)
ax.set_ylabel("AOI coverage (%)", fontsize=10)
ax.set_title("Coverage curve — AOI covered by basins with p ≥ τ'", fontsize=10)
ax.legend(fontsize=9)
ax.grid(True, which="both", alpha=0.25)

plt.tight_layout()
plt.savefig(OUT_DIR / "step7_basin_statistics.png", dpi=150,
            facecolor="white", bbox_inches="tight")
print(f"Saved → {OUT_DIR / 'step7_basin_statistics.png'}")
plt.show()